In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [26]:
df = pd.read_csv('../artifacts/raw_data/train_data.csv')
df.head()

,Booking_ID,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,arrival_year,arrival_month,arrival_date,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
0,INN25630,2,1,2,1,Meal Plan 1,0,Room_Type 1,26,2017,10,17,Online,0,0,0,161.00,0,Not_Canceled
1,INN14474,2,1,1,1,Meal Plan 1,0,Room_Type 1,98,2018,7,16,Online,0,0,0,121.50,2,Not_Canceled
2,INN23721,2,0,0,3,Meal Plan 1,0,Room_Type 1,433,2018,9,8,Offline,0,0,0,70.00,0,Canceled
3,INN05844,2,0,2,5,Meal Plan 1,0,Room_Type 1,195,2018,8,8,Offline,0,0,0,72.25,0,Not_Canceled
4,INN18710,1,0,0,2,Meal Plan 1,0,Room_Type 1,188,2018,6,15,Offline,0,0,0,130.00,0,Canceled


In [27]:
df.drop('Booking_ID', axis=1, inplace=True)
df.head()

,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,arrival_year,arrival_month,arrival_date,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
0,2,1,2,1,Meal Plan 1,0,Room_Type 1,26,2017,10,17,Online,0,0,0,161.00,0,Not_Canceled
1,2,1,1,1,Meal Plan 1,0,Room_Type 1,98,2018,7,16,Online,0,0,0,121.50,2,Not_Canceled
2,2,0,0,3,Meal Plan 1,0,Room_Type 1,433,2018,9,8,Offline,0,0,0,70.00,0,Canceled
3,2,0,2,5,Meal Plan 1,0,Room_Type 1,195,2018,8,8,Offline,0,0,0,72.25,0,Not_Canceled
4,1,0,0,2,Meal Plan 1,0,Room_Type 1,188,2018,6,15,Offline,0,0,0,130.00,0,Canceled


In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 29020 entries, 0 to 29019
Data columns (total 18 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   no_of_adults                          29020 non-null  int64  
 1   no_of_children                        29020 non-null  int64  
 2   no_of_weekend_nights                  29020 non-null  int64  
 3   no_of_week_nights                     29020 non-null  int64  
 4   type_of_meal_plan                     29020 non-null  str    
 5   required_car_parking_space            29020 non-null  int64  
 6   room_type_reserved                    29020 non-null  str    
 7   lead_time                             29020 non-null  int64  
 8   arrival_year                          29020 non-null  int64  
 9   arrival_month                         29020 non-null  int64  
 10  arrival_date                          29020 non-null  int64  
 11  market_segment_type       

In [29]:
cat_cols = df.select_dtypes(include='object').columns
num_cols = df.select_dtypes(include=np.number).columns

print(f"Categorical columns: {cat_cols}")
print(f"Numerical columns: {num_cols}")

Categorical columns: Index(['type_of_meal_plan', 'room_type_reserved', 'market_segment_type',
       'booking_status'],
      dtype='str')
Numerical columns: Index(['no_of_adults', 'no_of_children', 'no_of_weekend_nights',
       'no_of_week_nights', 'required_car_parking_space', 'lead_time',
       'arrival_year', 'arrival_month', 'arrival_date', 'repeated_guest',
       'no_of_previous_cancellations', 'no_of_previous_bookings_not_canceled',
       'avg_price_per_room', 'no_of_special_requests'],
      dtype='str')


In [39]:
from sklearn.preprocessing import LabelEncoder

encoders = {}
mappings = {}

data = df.copy()

for col in cat_cols:
    
    le = LabelEncoder()

    data[col] = le.fit_transform(data[col])

    # store encoder
    encoders[col] = le

    # store mappings
    mappings[col] = dict(
        zip(le.classes_, le.transform(le.classes_))
    )

mappings

{'type_of_meal_plan': {'Meal Plan 1': np.int64(0),
  'Meal Plan 2': np.int64(1),
  'Meal Plan 3': np.int64(2),
  'Not Selected': np.int64(3)},
 'room_type_reserved': {'Room_Type 1': np.int64(0),
  'Room_Type 2': np.int64(1),
  'Room_Type 3': np.int64(2),
  'Room_Type 4': np.int64(3),
  'Room_Type 5': np.int64(4),
  'Room_Type 6': np.int64(5),
  'Room_Type 7': np.int64(6)},
 'market_segment_type': {'Aviation': np.int64(0),
  'Complementary': np.int64(1),
  'Corporate': np.int64(2),
  'Offline': np.int64(3),
  'Online': np.int64(4)},
 'booking_status': {'Canceled': np.int64(0), 'Not_Canceled': np.int64(1)}}

In [36]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X = add_constant(data)
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values,i) for i in range(X.shape[1])]
vif_data

,feature,VIF
0,const,3.585767e+07
1,no_of_adults,1.295419e+00
2,no_of_children,1.257749e+00
3,no_of_weekend_nights,1.062324e+00
4,no_of_week_nights,1.086849e+00
5,type_of_meal_plan,1.155976e+00
6,required_car_parking_space,1.036866e+00
7,room_type_reserved,1.530014e+00
8,lead_time,1.440424e+00
9,arrival_year,1.297431e+00


In [37]:
from imblearn.over_sampling import SMOTE

X = data.drop(columns='booking_status')
y = data["booking_status"]

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)
y_resampled.value_counts()

booking_status
1    19551
0    19551
Name: count, dtype: int64

In [50]:
X_resampled.shape

(39102, 17)

In [38]:
from sklearn.preprocessing import StandardScaler

standard_scaler = StandardScaler()
X_train_scaled = standard_scaler.fit_transform(X_resampled)

In [44]:
test_data = pd.read_csv('../artifacts/raw_data/test_data.csv')
test_data.drop('Booking_ID', axis=1, inplace=True)


for col in cat_cols:

    le = encoders[col]

    # replace unseen values
    test_data[col] = test_data[col].apply(
        lambda x: x if x in le.classes_ else 'Unknown'
    )

    # add Unknown if not already present
    if 'Unknown' not in le.classes_:
        le.classes_ = np.append(le.classes_, 'Unknown')

    test_data[col] = le.transform(test_data[col])

X_test = test_data.drop(columns='booking_status')
y_test = test_data["booking_status"]
X_test_scaled = standard_scaler.transform(X_test)
X_test_scaled

array([[ 0.32178002,  2.33949345,  0.26875318, ..., -0.07500092,
        -0.44656582,  3.38622261],
       [ 0.32178002, -0.25043529,  0.26875318, ..., -0.07500092,
        -1.63937253, -0.69008789],
       [ 0.32178002, -0.25043529,  1.44845535, ..., -0.07500092,
        -0.29649093,  0.66868227],
       ...,
       [ 0.32178002, -0.25043529, -0.91094899, ..., -0.07500092,
         0.99646191, -0.69008789],
       [ 0.32178002, -0.25043529,  0.26875318, ..., -0.07500092,
        -0.59086859, -0.69008789],
       [-1.64819686, -0.25043529,  0.26875318, ..., -0.07500092,
        -0.06618371,  0.66868227]], shape=(7255, 17))

In [52]:
X_test_scaled

array([[ 0.32178002,  2.33949345,  0.26875318, ..., -0.07500092,
        -0.44656582,  3.38622261],
       [ 0.32178002, -0.25043529,  0.26875318, ..., -0.07500092,
        -1.63937253, -0.69008789],
       [ 0.32178002, -0.25043529,  1.44845535, ..., -0.07500092,
        -0.29649093,  0.66868227],
       ...,
       [ 0.32178002, -0.25043529, -0.91094899, ..., -0.07500092,
         0.99646191, -0.69008789],
       [ 0.32178002, -0.25043529,  0.26875318, ..., -0.07500092,
        -0.59086859, -0.69008789],
       [-1.64819686, -0.25043529,  0.26875318, ..., -0.07500092,
        -0.06618371,  0.66868227]], shape=(7255, 17))

In [47]:

from sklearn.metrics import classification_report, f1_score
from lightgbm import LGBMClassifier



clf = LGBMClassifier(random_state=42) 
clf.fit(X_train_scaled, y_resampled)

print("Training Classification Report:")
y_pred = clf.predict(X_train_scaled)
print(classification_report(y_resampled, y_pred))
print("Training F1 Score:", f1_score(y_resampled, y_pred))

print("\n\nTest Classification Report:")
y_pred = clf.predict(X_test_scaled)
print(classification_report(y_test, y_pred))
print("Test F1 Score:", f1_score(y_test, y_pred))

[LightGBM] [Info] Number of positive: 19551, number of negative: 19551
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001543 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 669
[LightGBM] [Info] Number of data points in the train set: 39102, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.88      0.90     19551
           1       0.89      0.92      0.90     19551

    accuracy                           0.90     39102
   macro avg       0.90      0.90      0.90     39102
weighted avg       0.90      0.90      0.90     39102

Training F1 Score: 0.9015210559396606


Test Classification Report:
              precision    recall  f1-score   support

           0    

In [48]:
X_resampled

,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,arrival_year,arrival_month,arrival_date,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests
0,2,1,2,1,0,0,0,26,2017,10,17,4,0,0,0,161.000000,0
1,2,1,1,1,0,0,0,98,2018,7,16,4,0,0,0,121.500000,2
2,2,0,0,3,0,0,0,433,2018,9,8,3,0,0,0,70.000000,0
3,2,0,2,5,0,0,0,195,2018,8,8,3,0,0,0,72.250000,0
4,1,0,0,2,0,0,0,188,2018,6,15,3,0,0,0,130.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39097,2,0,1,2,0,0,3,189,2018,10,28,4,0,0,0,109.800000,0
39098,2,1,0,1,0,0,0,11,2018,11,5,4,0,0,0,150.000000,1
39099,2,0,0,2,0,0,0,308,2018,11,25,3,0,0,0,52.000000,0
39100,2,0,0,2,1,0,0,377,2018,10,14,4,0,0,0,115.000000,1
